# 700 · Data & persistence — procedural layer

**Mnemonic:** SEVEN normal forms to tame your tables.

**Codes:** 703, 716, 720, 730, 741, 757, 761, 773, 780, 792.

**Method:** for every code cell — read it, PREDICT the exact output, run it, compare. A wrong prediction is the lesson: reread until you can predict it cold. Run cells top to bottom in ONE kernel — the sqlite connection created in 703 is reused all the way down. Only the two cells marked `# INTENDED ERROR — read the traceback` are supposed to raise.

## 703 · SELECT FROM WHERE

Core query shape: FROM picks tables, WHERE filters rows, SELECT projects columns — logically evaluated in that order.

*A fishmonger works right to left: nets the whole lake (FROM), throws back the small fish (WHERE), then fillets only the cuts you ordered (SELECT).*

**Watch for:** written order vs logical order — you type SELECT first, but the engine runs FROM, then WHERE, then SELECT. Predict which of the 4 rows survive the WHERE, and which columns each printed tuple contains.

In [1]:
import sqlite3

conn = sqlite3.connect(":memory:")   # ONE connection, reused by every sqlite cell below
conn.isolation_level = None          # autocommit; we will issue BEGIN/COMMIT by hand in 730
cur = conn.cursor()

cur.execute("CREATE TABLE users (id INTEGER PRIMARY KEY, name TEXT, age INTEGER)")
cur.executemany("INSERT INTO users VALUES (?, ?, ?)", [
    (1, "ada", 36),
    (2, "grace", 45),
    (3, "linus", 29),
    (4, "edsger", 51),
])

# logical order: FROM users -> WHERE age >= 35 -> SELECT name, age
for row in cur.execute("SELECT name, age FROM users WHERE age >= 35"):
    print(row)

('ada', 36)
('grace', 45)
('edsger', 51)


## 716 · EXPLAIN query plan

Shows the optimizer's chosen plan — scans, join order and algorithms, row estimates; the ANALYZE variant executes and adds real numbers.

*X-ray glasses for your query: put them on and its skeleton appears — a fat Seq Scan femur right where you swore an index lived.*

**Watch for:** EXPLAIN vs EXPLAIN ANALYZE — plain EXPLAIN only estimates; ANALYZE really runs the query. In SQLite the tell is one word: `SCAN` (read every row) vs `SEARCH ... USING INDEX` (jump straight there). Predict both plan lines before running.

In [2]:
cur.execute("CREATE TABLE events (id INTEGER PRIMARY KEY, city TEXT, amount INTEGER)")
cities = ["lisbon", "porto", "braga", "faro"]
cur.executemany("INSERT INTO events VALUES (?, ?, ?)",
                ((i, cities[i % 4], i * 3) for i in range(5000)))

query = "SELECT COUNT(*), SUM(amount) FROM events WHERE city = 'porto'"

plan_before = cur.execute("EXPLAIN QUERY PLAN " + query).fetchone()[3]
cur.execute("CREATE INDEX idx_events_city ON events(city)")
plan_after = cur.execute("EXPLAIN QUERY PLAN " + query).fetchone()[3]

print("before index:", plan_before)   # SCAN = touch all 5000 rows
print("after  index:", plan_after)    # SEARCH ... USING INDEX = only the matches
print("rows matched, total amount:", cur.execute(query).fetchone())

before index: SCAN events
after  index: SEARCH events USING INDEX idx_events_city (city=?)
rows matched, total amount: (1250, 9371250)


## 720 · Normalization

Structuring tables so every fact is stored exactly once, removing redundancy by decomposing along functional dependencies.

*Marie Kondo for data: every fact gets exactly one drawer; a duplicated fact sparks no joy and is folded away into its own table, leaving a key behind as the receipt.*

**Watch for:** denormalization is the deliberate reverse — reintroducing duplication for read speed once you know the cost. First cell shows the cost: the same fact stored 3 times can disagree with itself (update anomaly). Second cell: normalized, one UPDATE fixes every order. Predict ana's city per order in both cells.

In [3]:
# denormalized: customer_city is repeated on every order row
cur.execute("CREATE TABLE orders_denorm (order_id INTEGER PRIMARY KEY,"
            " customer TEXT, customer_city TEXT, amount INTEGER)")
cur.executemany("INSERT INTO orders_denorm VALUES (?, ?, ?, ?)", [
    (101, "ana",   "lisbon", 30),
    (102, "ana",   "lisbon", 50),
    (103, "ana",   "lisbon", 20),
    (104, "bruno", "porto",  40),
])

# ana moves to porto, but a buggy UPDATE misses order 103...
cur.execute("UPDATE orders_denorm SET customer_city = 'porto'"
            " WHERE customer = 'ana' AND order_id < 103")

print("ana's city according to each of her order rows:")
for row in cur.execute("SELECT order_id, customer_city FROM orders_denorm"
                       " WHERE customer = 'ana' ORDER BY order_id"):
    print(" ", row)
cities = cur.execute("SELECT DISTINCT customer_city FROM orders_denorm"
                     " WHERE customer = 'ana'").fetchall()
print("distinct answers to 'where does ana live?':", [c[0] for c in cities])
# the database now holds two contradictory versions of one fact: an UPDATE anomaly

ana's city according to each of her order rows:
  (101, 'porto')
  (102, 'porto')
  (103, 'lisbon')
distinct answers to 'where does ana live?': ['porto', 'lisbon']


In [4]:
# normalized: the city fact lives in exactly ONE row of customers
cur.execute("CREATE TABLE customers (id INTEGER PRIMARY KEY, name TEXT, city TEXT)")
cur.execute("CREATE TABLE orders_norm (order_id INTEGER PRIMARY KEY,"
            " customer_id INTEGER, amount INTEGER)")
cur.executemany("INSERT INTO customers VALUES (?, ?, ?)",
                [(1, "ana", "lisbon"), (2, "bruno", "porto")])
cur.executemany("INSERT INTO orders_norm VALUES (?, ?, ?)",
                [(101, 1, 30), (102, 1, 50), (103, 1, 20), (104, 2, 40)])

cur.execute("UPDATE customers SET city = 'porto' WHERE name = 'ana'")  # ONE row changed

print("ana's city according to each of her orders (via JOIN):")
for row in cur.execute("SELECT o.order_id, c.city FROM orders_norm o"
                       " JOIN customers c ON c.id = o.customer_id"
                       " WHERE c.name = 'ana' ORDER BY o.order_id"):
    print(" ", row)
# one UPDATE, zero anomalies: every order agrees, because the fact exists once

ana's city according to each of her orders (via JOIN):
  (101, 'porto')
  (102, 'porto')
  (103, 'porto')


## 730 · Transaction

A group of statements executed as one all-or-nothing unit: COMMIT makes all of it visible; ROLLBACK erases every trace.

*A magician's tablecloth yank: either the whole new table setting appears in one motion, or the trick fails and every plate is exactly where it was. The audience never sees half-moved cutlery.*

**Watch for:** single autocommit statement vs transaction — each statement is already its own tiny transaction; BEGIN binds several into one fate. Predict all four printed counts, especially the count AFTER two successful inserts but a ROLLBACK.

In [5]:
cur.execute("CREATE TABLE accounts (id INTEGER PRIMARY KEY, name TEXT)")

def n_accounts():
    return cur.execute("SELECT COUNT(*) FROM accounts").fetchone()[0]

print("count at start          :", n_accounts())

cur.execute("BEGIN")
cur.execute("INSERT INTO accounts VALUES (1, 'alice')")
cur.execute("INSERT INTO accounts VALUES (2, 'bob')")
print("count inside transaction:", n_accounts())
try:
    cur.execute("INSERT INTO accounts VALUES (1, 'mallory')")  # id 1 again -> error
except sqlite3.IntegrityError as err:
    print("error mid-transaction   :", err)
    cur.execute("ROLLBACK")            # all-or-nothing: alice and bob vanish too
print("count after ROLLBACK    :", n_accounts())

cur.execute("BEGIN")
cur.execute("INSERT INTO accounts VALUES (1, 'alice')")
cur.execute("INSERT INTO accounts VALUES (2, 'bob')")
cur.execute("COMMIT")                  # the tablecloth trick lands: both appear at once
print("count after COMMIT      :", n_accounts())

count at start          : 0
count inside transaction: 2
error mid-transaction   : UNIQUE constraint failed: accounts.id
count after ROLLBACK    : 0
count after COMMIT      : 2


## 741 · Key-value store

Stores an opaque value under a unique key; you can GET, SET, and DELETE by exact key but never query inside the value.

*A wall of gym lockers: present exactly key #417 and get your bag. Ask the attendant 'which lockers contain red jackets?' and he just stares — lockers are opaque; only keys open anything.*

**Watch for:** document store vs KV — documents are queryable and indexable by inner fields; a KV value is a sealed blob. Predict what `get` returns after the `delete`. Redis and DynamoDB are this exact idea, industrialized.

In [6]:
import json

class KVStore:
    # toy key-value store: opaque JSON blobs filed under exact keys
    def __init__(self):
        self._data = {}
    def put(self, key, value):
        self._data[key] = json.dumps(value)   # value goes in as a sealed text blob
    def get(self, key):
        raw = self._data.get(key)
        return None if raw is None else json.loads(raw)
    def delete(self, key):
        return self._data.pop(key, None) is not None

kv = KVStore()
kv.put("session:abc", {"user": 7, "ttl": 3600})
kv.put("session:xyz", {"user": 9})
print("get    session:abc ->", kv.get("session:abc"))
print("delete session:xyz ->", kv.delete("session:xyz"))
print("get    session:xyz ->", kv.get("session:xyz"))
print("keys left:", sorted(kv._data))
# no WHERE, no query-by-value: 'which sessions belong to user 7?' is unanswerable
# without scanning every locker. Redis / DynamoDB = this idea, industrialized.

get    session:abc -> {'user': 7, 'ttl': 3600}
delete session:xyz -> True
get    session:xyz -> None
keys left: ['session:abc']


## 757 · LRU and LFU eviction

When full, the cache evicts by recency (LRU: least recently used) or by frequency (LFU: least frequently used) to make room.

*A stuffed closet: LRU tosses whatever you wore longest ago; LFU counts wears and tosses the least-worn. The tux worn once last night survives LRU's purge but is first out under LFU.*

**Watch for:** TTL removes entries because they are old; eviction removes them because the cache is full. With maxsize=2 and the call pattern a, b, a, c, b — predict exactly which calls print `computing` (miss) and the final hits/misses/currsize.

In [7]:
import functools

@functools.lru_cache(maxsize=2)
def load(key):
    print(f"   computing {key!r} (cache miss)")
    return key.upper()

for key in ["a", "b", "a", "c", "b"]:
    print(f"load({key!r}) -> {load(key)}")

info = load.cache_info()
print(info)
print("evictions so far:", info.misses - info.currsize)
# trace: a miss, b miss, a HIT (a becomes most-recent), c miss -> evicts b (LRU),
# b miss again -> evicts a. Recency, not alphabet, decides who gets tossed.

   computing 'a' (cache miss)
load('a') -> A
   computing 'b' (cache miss)
load('b') -> B
load('a') -> A
   computing 'c' (cache miss)
load('c') -> C
   computing 'b' (cache miss)
load('b') -> B
CacheInfo(hits=1, misses=4, maxsize=2, currsize=2)
evictions so far: 2


## 761 · JSON

Ubiquitous text format of objects, arrays, strings, numbers, booleans, null — human-readable and schema-free; no comments, dates, or safe ints past 2^53.

*Jason, the friendly courier every country understands: travels light, speaks plainly, checks no ID (schema). But he can't pronounce dates — he hands them over scribbled as strings — and drops any coin bigger than 2^53.*

**Watch for:** JSON vs JavaScript object literal — JSON is stricter: double-quoted keys, no comments, no trailing commas. Cell 1: predict the serialized text (what happens to True and None?). Cell 2 raises on purpose — Jason cannot pronounce dates. Cell 3 is the standard fix.

In [8]:
import json

doc = {"id": 7, "tags": ["a", "b"], "meta": {"ok": True, "score": None}}

text = json.dumps(doc)     # python -> JSON text
back = json.loads(text)    # JSON text -> python

print("serialized :", text)
print("round-trip equal:", back == doc)
# note in the text: True became true, None became null, keys grew double quotes

serialized : {"id": 7, "tags": ["a", "b"], "meta": {"ok": true, "score": null}}
round-trip equal: True


In [9]:
# INTENDED ERROR — read the traceback
import json
from datetime import datetime

json.dumps({"when": datetime.now()})   # JSON has no date type -> TypeError

TypeError: Object of type datetime is not JSON serializable

In [10]:
from datetime import datetime

stamp = datetime(2026, 7, 11, 9, 30, 0)          # fixed value, deterministic output
text = json.dumps({"when": stamp.isoformat()})   # the fix: dates travel as ISO-8601 strings
print("serialized:", text)
print("and back  :", json.loads(text)["when"], "<- comes back as a plain string, by convention")

serialized: {"when": "2026-07-11T09:30:00"}
and back  : 2026-07-11T09:30:00 <- comes back as a plain string, by convention


## 773 · Write-ahead log

Every change is appended and fsynced to a sequential log before data pages are touched; replaying the log after a crash restores committed work.

*The captain's iron rule: write the maneuver in the logbook before touching the sails. When the storm (crash) hits mid-turn, the surviving mate replays the logbook line by line and the ship ends exactly where the entries promised.*

**Watch for:** WAL vs audit trail — the WAL is physical, engine-internal, recycled after checkpoints; an audit trail is business-level and kept for humans. Predict the pre-crash state after the 5 ops (note the DEL), then confirm replay rebuilds it exactly.

In [11]:
log = []      # the write-ahead log: append-only, always written FIRST
state = {}    # the 'data pages': fast, but lost in a crash

def apply_op(target, op):
    kind, key, *rest = op.split()
    if kind == "SET":
        target[key] = int(rest[0])
    elif kind == "DEL":
        target.pop(key, None)

for op in ["SET x 1", "SET y 2", "SET x 5", "DEL y", "SET z 9"]:
    log.append(op)        # 1) logbook first (a real engine fsyncs here)
    apply_op(state, op)   # 2) only then touch the sails

print("state before crash:", state)
pre_crash = dict(state)

state = None              # CRASH — in-memory data pages are gone
recovered = {}
for op in log:            # recovery = replay the log, in order, from the start
    apply_op(recovered, op)

print("recovered from log:", recovered)
print("recovered == pre-crash:", recovered == pre_crash)

state before crash: {'x': 5, 'z': 9}
recovered from log: {'x': 5, 'z': 9}
recovered == pre-crash: True


## 780 · OLTP vs OLAP

OLTP: many tiny concurrent reads/writes touching single rows (row stores). OLAP: few huge scans aggregating millions of rows (column stores).

*Downstairs, checkout lanes beep thousands of tiny basket updates a minute (OLTP). Upstairs at midnight, one accountant reads EVERY receipt of the year to ask how bananas did (OLAP). Different rooms, different filing.*

**Watch for:** HTAP hybrids try both in one system; the classic split is row store for OLTP, columnar warehouse for OLAP. Same 50,000 records, two layouts — predict which layout sums one column faster, then check the ratio. Exact times vary per machine; the winner does not.

In [12]:
import timeit

N = 50_000
rows = [(i, i % 7, i * 2) for i in range(N)]      # row layout: one tuple per record
cols = {                                          # column layout: one list per field
    "id":       list(range(N)),
    "category": [i % 7 for i in range(N)],
    "value":    [i * 2 for i in range(N)],
}

t_rows = timeit.timeit(lambda: sum(r[2] for r in rows), number=20)
t_cols = timeit.timeit(lambda: sum(cols["value"]),      number=20)

print(f"sum 'value' via rows    : {t_rows:.4f} s  (20 scans)")
print(f"sum 'value' via columns : {t_cols:.4f} s  (20 scans)")
print(f"columnar scan wins by ~{t_rows / t_cols:.0f}x")
# the accountant wants ONE column of every receipt: a contiguous list beats
# hopping through 50,000 tuples. That is why warehouses store by column.

sum 'value' via rows    : 0.0270 s  (20 scans)
sum 'value' via columns : 0.0047 s  (20 scans)
columnar scan wins by ~6x


## 792 · UNIQUE constraint

Guarantees no two rows share the same value(s) in the constrained column(s), enforced by a unique index; NULLs typically don't count as duplicates.

*The marathon official at the start line scans bib numbers against a sorted roster (unique index): a second runner in bib 1042 is yanked off the course before the gun — no two racers ever share a number mid-race.*

**Watch for:** primary key vs UNIQUE — a PK is UNIQUE plus NOT NULL and one per table; UNIQUE constraints can be many and usually admit NULLs. Cell 1 inserts cleanly. Cell 2 raises on purpose — predict the exact exception type before running.

In [13]:
cur.execute("CREATE TABLE signups (id INTEGER PRIMARY KEY, email TEXT, UNIQUE(email))")
cur.execute("INSERT INTO signups VALUES (1, 'ada@example.com')")
print("first insert OK:", cur.execute("SELECT * FROM signups").fetchall())

first insert OK: [(1, 'ada@example.com')]


In [14]:
# INTENDED ERROR — read the traceback
cur.execute("INSERT INTO signups VALUES (2, 'ada@example.com')")  # same email, different id
# the id is new, but the UNIQUE(email) roster already has this bib number

IntegrityError: UNIQUE constraint failed: signups.email